In [2]:
import pandas as pd
import numpy as np
from itertools import product
from scipy.stats import norm
from scipy.optimize import minimize
import matplotlib.pyplot as plt

In [3]:
# fais un dataframe avec les données brutes
data_rating_raw = pd.read_csv("../data/credit_ratings/sectors_quater.csv")

In [4]:
df = data_rating_raw.copy()


Importation et nettoyage des données

In [5]:
df.head()

,year_quarter,obligor_name,rating_agency_name,rating,rating_action_date,legal_entity_identifier,year_month,year,pays,nace,next_rating
0,2022-07-01,"06 ENVIRONMENTAL, LLC",Egan-Jones Ratings Company,B,2022-07-22,NaN,2022-07-01,2022.0,NaN,RU,B
1,2021-07-01,"11065220 Canada, Inc.",Fitch Ratings,B,2021-08-18,549300ETSKJL315VDV79,2021-08-01,2021.0,NaN,RU,D
2,2019-04-01,"18 Fremont Street Acquisition, LLC",Moody's Investors Service,B,2019-06-06,5493000QK0X0K188BR60,2019-06-01,2019.0,US,RU,C
3,2019-07-01,"18 Fremont Street Acquisition, LLC",Moody's Investors Service,B,2019-07-01,5493000QK0X0K188BR60,2019-07-01,2019.0,US,RU,C
4,2019-10-01,"18 Fremont Street Acquisition, LLC",Moody's Investors Service,B,2019-10-01,5493000QK0X0K188BR60,2019-10-01,2019.0,US,RU,C


In [6]:
df["pays"].unique()

<StringArray>
[ nan, 'US', 'GB', 'DE', 'CA', 'SE', 'CH', 'RU', 'IN', 'ID', 'ES', 'FR', 'TW',
 'JP', 'NO', 'NL', 'BR', 'AE', 'NZ', 'KY', 'IE', 'BE', 'LU', 'IT', 'CO', 'DK',
 'GI', 'AU', 'JE', 'AR', 'CL', 'CY', 'BM', 'VN', 'MX', 'CN', 'HK', 'AT', 'PA',
 'PH', 'ZA', 'PT', 'IL', 'MH', 'NG', 'OM', 'RO', 'LC', 'TR', 'FI', 'SG', 'VG',
 'MY', 'PE', 'MU', 'BS', 'IS', 'PY', 'GE', 'GG', 'IM', 'TT', 'BG', 'PL', 'LR',
 'CW', 'GL', 'LK', 'CZ', 'SA']
Length: 70, dtype: str

In [12]:
#créeons une fonction pour venir nettoyer et préparer les données :
def  clean_data(df : pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    #commencons par transformer les variables de temps en séries temporelles : 
    df["year_quarter"] = pd.to_datetime(df["year_quarter"], errors = "coerce")  
    df["rating_action_date"] = pd.to_datetime(df["rating_action_date"], errors = "coerce")
    df["year_month"]= pd.to_datetime(df["year_month"], errors ="coerce")

    #forcer les strings sur les variables qualitatives: 
    for c in ["rating_agency_name", "obligor_name", "pays", "rating", "legal_entity_identifier", "nace"]: 
        df[c] = df[c].astype(str).strip()

    #nettoyage sur secteur NACE : 
    df["nace"] = df["nace"].str.upper()
    df['rating'] = df['rating'].str.upper()

    return df

def quick_analysis(df : pd.DataFrame):
    print("----DIAGNOSTIC----")
    print("Nombre de lignes: ", len(df))
    print("nombre d'entreprises : ", df["obligor_name"].nunique())
    print("Nombre d'agences : ", df['rating_agency_name'].nunique())
    print("Nombre de secteurs :" , df['nace'].nunique())
    print("Période de temps : ", df["year_quarter"].min(), " -> ", df["year_quarter"].max())

    print("\n top agences", df['rating_agency_name'].value_counts().head(5))
    print("\n top ratings", df['rating'].value_counts().head(5))

    print("\n top secteurs", df['nace'].value_counts().head(10))
    for c in df.select_dtypes(include="object").columns:
        print(f"{c}: {df[c].unique()[:5]}...")  # show first 5 unique values



In [13]:
#Appel de clean data : 
clean_data(df)

AttributeError: 'Series' object has no attribute 'strip'